# A/B Hypothesis Testing: Insurance Risk Analysis

## Objective
Statistically validate or reject key hypotheses about risk drivers across provinces, zip codes, and gender.
This forms the evidence base for ACIS's new segmentation and pricing strategy.

## Hypotheses to Test
1. H₀: There are no risk differences across provinces
2. H₀: There are no risk differences between zip codes
3. H₀: There is no significant margin difference between zip codes
4. H₀: There is no significant risk difference between Women and Men

## Significance Level
α = 0.05 (reject H₀ when p < 0.05)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import DataLoader
from src.hypothesis_tests import (
    HypothesisTest,
    test_risk_by_province,
    test_risk_by_zipcode,
    test_margin_by_zipcode,
    test_risk_by_gender,
    calculate_effect_size,
    calculate_power,
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully')

## 1. Load and Prepare Data

In [ ]:
# Load data
data_path = '../data/insurance_data.csv'
loader = DataLoader(data_path)

try:
    df = loader.load_data()
    print(f'Data loaded successfully: {df.shape}')
    
    # Create derived metrics
    df = loader.create_derived_metrics(df)
    print('Derived metrics created')
    
except FileNotFoundError:
    print(f'Data file not found at {data_path}')
    df = None

## 2. Hypothesis 1: Risk Differences Across Provinces

In [ ]:
if df is not None and 'Province' in df.columns:
    print('HYPOTHESIS 1: Risk Differences Across Provinces')
    print('='*60)
    print('H₀: There are no risk differences across provinces')
    print('H₁: There are risk differences across provinces')
    print('\nTest: Chi-Squared Test')
    print('KPI: Claim Frequency (binary: claim occurred or not)')
    print('\nRunning test...')
    
    result_h1 = test_risk_by_province(df, alpha=0.05)
    
    print(f'\nResults:')
    print(f'Chi-Squared Statistic: {result_h1["statistic"]:.4f}')
    print(f'p-value: {result_h1["p_value"]:.6f}')
    print(f'Degrees of Freedom: {result_h1["dof"]}')
    print(f'Significance Level (α): {result_h1["alpha"]}')
    print(f'\nDecision: {"REJECT H₀" if result_h1["reject_null"] else "FAIL TO REJECT H₀"}')
    
    if result_h1['reject_null']:
        print('\nConclusion: There ARE statistically significant risk differences across provinces.')
        print('Business Implication: Regional risk adjustments to premiums may be warranted.')
    else:
        print('\nConclusion: No statistically significant risk differences across provinces.')
        print('Business Implication: Province-based pricing adjustments may not be justified.')

In [ ]:
if df is not None and 'Province' in df.columns:
    # Contingency table
    print('\nContingency Table:')
    print(result_h1['contingency_table'])
    
    # Claim frequency by province
    claim_freq_by_prov = df.groupby('Province')['ClaimIndicator'].agg(['sum', 'count', 'mean'])
    claim_freq_by_prov.columns = ['Claims', 'Total_Policies', 'Claim_Frequency']
    print('\nClaim Frequency by Province:')
    print(claim_freq_by_prov)

## 3. Hypothesis 2: Risk Differences Across Zip Codes

In [ ]:
if df is not None and 'PostalCode' in df.columns:
    print('\nHYPOTHESIS 2: Risk Differences Across Zip Codes')
    print('='*60)
    print('H₀: There are no risk differences between zip codes')
    print('H₁: There are risk differences between zip codes')
    print('\nTest: Chi-Squared Test')
    print('KPI: Claim Frequency (binary: claim occurred or not)')
    print('\nRunning test...')
    
    result_h2 = test_risk_by_zipcode(df, alpha=0.05)
    
    print(f'\nResults:')
    print(f'Chi-Squared Statistic: {result_h2["statistic"]:.4f}')
    print(f'p-value: {result_h2["p_value"]:.6f}')
    print(f'Degrees of Freedom: {result_h2["dof"]}')
    print(f'\nDecision: {"REJECT H₀" if result_h2["reject_null"] else "FAIL TO REJECT H₀"}')
    
    if result_h2['reject_null']:
        print('\nConclusion: There ARE statistically significant risk differences across zip codes.')
        print('Business Implication: Zip code-based segmentation could improve pricing accuracy.')
    else:
        print('\nConclusion: No statistically significant risk differences across zip codes.')
        print('Business Implication: Zip code may not be a strong risk predictor.')
else:
    print('PostalCode column not found in data')

## 4. Hypothesis 3: Margin Differences Across Zip Codes

In [ ]:
if df is not None and 'PostalCode' in df.columns:
    print('\nHYPOTHESIS 3: Margin Differences Across Zip Codes')
    print('='*60)
    print('H₀: There is no significant margin difference between zip codes')
    print('H₁: There is a significant margin difference between zip codes')
    print('\nTest: Independent Samples t-test')
    print('KPI: Margin (TotalPremium - TotalClaims)')
    print('\nRunning test...')
    
    result_h3 = test_margin_by_zipcode(df, alpha=0.05)
    
    print(f'\nResults:')
    print(f't-Statistic: {result_h3["statistic"]:.4f}')
    print(f'p-value: {result_h3["p_value"]:.6f}')
    print(f'Group A Mean Margin: R{result_h3["group_a_mean"]:,.2f}')
    print(f'Group B Mean Margin: R{result_h3["group_b_mean"]:,.2f}')
    print(f'\nDecision: {"REJECT H₀" if result_h3["reject_null"] else "FAIL TO REJECT H₀"}')
    
    if result_h3['reject_null']:
        print('\nConclusion: There IS a statistically significant margin difference.')
        print('Business Implication: Profitability varies significantly by zip code.')
    else:
        print('\nConclusion: No statistically significant margin difference.')
        print('Business Implication: Zip code does not significantly impact profitability.')
    
    # Effect size
    effect_size = calculate_effect_size(
        df[df['PostalCode'] == result_h3['test_name'].split()[-1]]['Margin'].values,
        df[df['PostalCode'] != result_h3['test_name'].split()[-1]]['Margin'].values
    )
    print(f'\nEffect Size (Cohen\'s d): {effect_size:.4f}')
else:
    print('PostalCode column not found in data')

## 5. Hypothesis 4: Risk Differences by Gender

In [ ]:
if df is not None and 'Gender' in df.columns:
    print('\nHYPOTHESIS 4: Risk Differences by Gender')
    print('='*60)
    print('H₀: There is no significant risk difference between Women and Men')
    print('H₁: There is a significant risk difference between Women and Men')
    print('\nTest: Chi-Squared Test')
    print('KPI: Claim Frequency (binary: claim occurred or not)')
    print('\nRunning test...')
    
    result_h4 = test_risk_by_gender(df, alpha=0.05)
    
    print(f'\nResults:')
    print(f'Chi-Squared Statistic: {result_h4["statistic"]:.4f}')
    print(f'p-value: {result_h4["p_value"]:.6f}')
    print(f'\nDecision: {"REJECT H₀" if result_h4["reject_null"] else "FAIL TO REJECT H₀"}')
    
    if result_h4['reject_null']:
        print('\nConclusion: There IS a statistically significant risk difference by gender.')
        print('Business Implication: Gender-based pricing adjustments may be warranted.')
    else:
        print('\nConclusion: No statistically significant risk difference by gender.')
        print('Business Implication: Gender-based pricing may not be justified.')
    
    # Claim frequency by gender
    claim_freq_by_gender = df.groupby('Gender')['ClaimIndicator'].agg(['sum', 'count', 'mean'])
    claim_freq_by_gender.columns = ['Claims', 'Total_Policies', 'Claim_Frequency']
    print('\nClaim Frequency by Gender:')
    print(claim_freq_by_gender)
else:
    print('Gender column not found in data')

## 6. Summary of All Hypothesis Tests

In [ ]:
if df is not None:
    # Create summary table
    results_summary = []
    
    if 'Province' in df.columns:
        results_summary.append({
            'Hypothesis': 'Risk by Province',
            'Test Type': 'Chi-Squared',
            'Statistic': f"{result_h1['statistic']:.4f}",
            'p-value': f"{result_h1['p_value']:.6f}",
            'Decision': 'Reject H₀' if result_h1['reject_null'] else 'Fail to Reject',
            'Significance': 'Yes' if result_h1['reject_null'] else 'No'
        })
    
    if 'PostalCode' in df.columns:
        results_summary.append({
            'Hypothesis': 'Risk by Zip Code',
            'Test Type': 'Chi-Squared',
            'Statistic': f"{result_h2['statistic']:.4f}",
            'p-value': f"{result_h2['p_value']:.6f}",
            'Decision': 'Reject H₀' if result_h2['reject_null'] else 'Fail to Reject',
            'Significance': 'Yes' if result_h2['reject_null'] else 'No'
        })
        
        results_summary.append({
            'Hypothesis': 'Margin by Zip Code',
            'Test Type': 't-test',
            'Statistic': f"{result_h3['statistic']:.4f}",
            'p-value': f"{result_h3['p_value']:.6f}",
            'Decision': 'Reject H₀' if result_h3['reject_null'] else 'Fail to Reject',
            'Significance': 'Yes' if result_h3['reject_null'] else 'No'
        })
    
    if 'Gender' in df.columns:
        results_summary.append({
            'Hypothesis': 'Risk by Gender',
            'Test Type': 'Chi-Squared',
            'Statistic': f"{result_h4['statistic']:.4f}",
            'p-value': f"{result_h4['p_value']:.6f}",
            'Decision': 'Reject H₀' if result_h4['reject_null'] else 'Fail to Reject',
            'Significance': 'Yes' if result_h4['reject_null'] else 'No'
        })
    
    summary_df = pd.DataFrame(results_summary)
    print('\nHYPOTHESIS TESTING SUMMARY')
    print('='*80)
    print(summary_df.to_string(index=False))

## 7. Business Recommendations

In [ ]:
print('\nBUSINESS RECOMMENDATIONS')
print('='*80)

if df is not None and 'Province' in df.columns and result_h1['reject_null']:
    print('\n1. PROVINCE-BASED SEGMENTATION')
    print('   Status: SIGNIFICANT DIFFERENCES FOUND')
    print('   Recommendation: Implement province-based premium adjustments')
    print('   Action: Develop regional pricing tiers based on loss ratios')

if df is not None and 'PostalCode' in df.columns and result_h2['reject_null']:
    print('\n2. ZIP CODE-BASED SEGMENTATION')
    print('   Status: SIGNIFICANT DIFFERENCES FOUND')
    print('   Recommendation: Implement zip code-based premium adjustments')
    print('   Action: Develop granular geographic pricing model')

if df is not None and 'PostalCode' in df.columns and result_h3['reject_null']:
    print('\n3. PROFITABILITY OPTIMIZATION')
    print('   Status: MARGIN DIFFERENCES FOUND')
    print('   Recommendation: Focus marketing on high-margin zip codes')
    print('   Action: Allocate marketing budget to profitable segments')

if df is not None and 'Gender' in df.columns and result_h4['reject_null']:
    print('\n4. GENDER-BASED PRICING')
    print('   Status: SIGNIFICANT DIFFERENCES FOUND')
    print('   Recommendation: Consider gender in risk assessment')
    print('   Action: Develop gender-adjusted pricing models')

print('\n5. NEXT STEPS')
print('   - Validate findings with additional data')
print('   - Develop predictive models for claim severity')
print('   - Design A/B tests for pricing changes')
print('   - Monitor impact on customer acquisition and retention')